In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM

import warnings
import massbalancemachine as mbm
import logging
from datetime import datetime
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import json 

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.utils import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

print(torch.cuda.is_available())   # True
print(torch.cuda.device_count())   # 0  ← wrong$

if torch.cuda.device_count()==0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")


In [ ]:
GLOB_DIR = Path(cfg.dataPath) / path_cache
BASE_DIR = Path(cfg.dataPath) / path_cache / f"TF_REGION"
print(f"Base directory for data: {BASE_DIR}")

# make sure the base directory exists
os.makedirs(BASE_DIR, exist_ok=True)

In [ ]:
MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc"),
}
# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

# Transform data to monthly format (run or load data):
paths_HMA = {
    "era5_climate_data":
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_HIGH_MOUNTAIN_ASIA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_HIGH_MOUNTAIN_ASIA.nc"),
}
# Check that all these files exists
for key, path in paths_HMA.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

vois_climate = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
]

vois_topographical = ["aspect", "slope", "svf"]

# Cross-regional modelling:

### Read stakes datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (SJM+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
dfs["11"]

In [ ]:
rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
num_measurements

### Monthly datasets:
Build monthly datasets for LSTM. 

In [ ]:
SOURCE_REGIONS = [
    'CH',
    'NOR',
    'ISL',
    'ALA',
    "CENTRALASIA",
]
experiment_region_groups = {
    # "CEU": ["FR", "IT_AT", "CH"],
    "HMA": ["CENTRALASIA", "SOUTHASIAWEST", "SOUTHASIAEAST"]
}
paths_multi = {
    "EU_US_CANADA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_EU_US_CANADA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_EU_US_CANADA.nc"),
    },
    "HMA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_HIGH_MOUNTAIN_ASIA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_HIGH_MOUNTAIN_ASIA.nc"),
    },
}

XREG_PAIRS = [
    # (source, target)
    ("CH", [
        "NOR",
        "ISL",
        "SJM",
        "ALA",
        "CENTRALASIA",
    ]),
    ("NOR", [
        "CH",
        "ISL",
        "SJM",
        "ALA",
        "CENTRALASIA",
    ]),
    ("ISL", [
        "CH",
        "NOR",
        "SJM",
        "ALA",
        "CENTRALASIA",
    ]),
    ("ALA", [
        "CH",
        "NOR",
        "ISL",
        "SJM",
        "CENTRALASIA",
    ]),
    ("CENTRALASIA", [
        "CH",
        "NOR",
        "ISL",
        "SJM",
        "ALA",
    ])
]

In [ ]:
# Step 1: collect all unique codes across all pairs
all_codes = sorted(
    {code.upper()
     for src, tgts in XREG_PAIRS
     for code in [src] + tgts})

run_flag_by_code = {
    'ALA': False,
    'CENTRALASIA': False,
    'CH': False,
    'ISL': False,
    'NOR': False,
    'SJM': False,
}
# Step 2: compute monthly data once per code
monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=vois_climate,
    vois_topographical=vois_topographical,
    region_codes=all_codes,
    run_flag_by_code=run_flag_by_code,
)

# Step 3: assemble train/test pairs from cache — no ERA5 reprocessing
res_xreg_by_source, split_info_by_source = prepare_xreg_pairs_from_cache(
    cfg=cfg,
    monthly_cache=monthly_cache,
    xreg_pairs=XREG_PAIRS,
)

### Finetuning glaciers:

#### Automatic picking:

In [ ]:
CLIMATE_COLS = [
    't2m', 'tp', 'slhf', 'sshf', 'ssrd', 'fal', 'str', 'ELEVATION_DIFFERENCE'
]
TOPO_COLS = ['aspect', 'slope', 'svf']

# define scalars:
scaler_m, scaler_s, scaler_joint = build_global_scalers_multi_source_simple(
    res_xreg_by_source,
    monthly_cols=CLIMATE_COLS,
    static_cols=TOPO_COLS,
)

blur_m, blur_s, blur_joint = estimate_global_bandwidths_simple(
    res_xreg_by_source,
    monthly_cols=CLIMATE_COLS,
    static_cols=TOPO_COLS,
    scaler_m=scaler_m,
    scaler_s=scaler_s,
    seed=cfg.seed,
)
bandwidths_m = [blur_m * 0.5, blur_m, blur_m * 2.0]
bandwidths_s = [blur_s * 0.5, blur_s, blur_s * 2.0]
print(
    f"Estimated blurs: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
)

In [ ]:
RECOMPUTE_SPLITS = False  # set to True to rerun the optimization
save_path = BASE_DIR / "glacier_splits_FT.json"

if not RECOMPUTE_SPLITS and save_path.exists():
    with open(save_path, "r") as f:
        output = json.load(f)
    splits = {
        src_region: {
            "pool_glaciers": data["pool_glaciers"],
            "holdout_glaciers": data["holdout_glaciers"],
            "actual_pool_frac": data["actual_pool_frac"],
            "sinkhorn_pool_vs_holdout": data["sinkhorn_pool_vs_holdout"],
            "sinkhorn_pool_vs_region": data["sinkhorn_pool_vs_region"],
            "sinkhorn_holdout_vs_region": data["sinkhorn_holdout_vs_region"],
            "blur_joint": data["blur_joint"],
            "best_seed": data["best_seed"],
            "best_score": data["best_score"],
        }
        for src_region, data in output.items()
    }
    print(f"Loaded splits from: {save_path}")
    for src_region, split in splits.items():
        print(f"\n{src_region}:")
        print(f"  Pool glaciers     : {split['pool_glaciers']}")
        print(f"  Holdout glaciers  : {split['holdout_glaciers']}")
        print(f"  Pool fraction     : {split['actual_pool_frac']:.1%}")
        print(f"  Sk(pool↔holdout)  : {split['sinkhorn_pool_vs_holdout']:.4f}")

else:
    splits = {}

    for src_region in SOURCE_REGIONS:
        print(f"\n{'='*50}")
        print(f"Splitting region: {src_region}")

        df_target = res_xreg_by_source[src_region]['df_train']

        split = split_pool_holdout_sinkhorn(
            df_region=df_target,
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS,
            scaler_m=scaler_m,
            scaler_s=scaler_s,
            blur_joint=blur_joint,
            pool_frac=0.2,
            n_restarts=50,
            seed=cfg.seed,
        )

        splits[src_region] = split

        print(f"\nPool glaciers     : {split['pool_glaciers']}")
        print(f"Holdout glaciers  : {split['holdout_glaciers']}")
        print(f"Pool fraction     : {split['actual_pool_frac']:.1%}")
        print(f"Sk(pool↔holdout)  : {split['sinkhorn_pool_vs_holdout']:.4f}")
        print(f"Sk(pool↔region)   : {split['sinkhorn_pool_vs_region']:.4f}")
        print(f"Sk(holdout↔region): {split['sinkhorn_holdout_vs_region']:.4f}")

    output = {
        src_region: {
            "pool_glaciers": split["pool_glaciers"],
            "holdout_glaciers": split["holdout_glaciers"],
            "actual_pool_frac": split["actual_pool_frac"],
            "sinkhorn_pool_vs_holdout": split["sinkhorn_pool_vs_holdout"],
            "sinkhorn_pool_vs_region": split["sinkhorn_pool_vs_region"],
            "sinkhorn_holdout_vs_region": split["sinkhorn_holdout_vs_region"],
            "blur_joint": split["blur_joint"],
            "best_seed": split["best_seed"],
            "best_score": split["best_score"],
        }
        for src_region, split in splits.items()
    }

    with open(save_path, "w") as f:
        json.dump(output, f, indent=2)

    print(f"\nAll splits saved to: {save_path}")


#### Final ft and hold-out glaciers:

In [ ]:
FT_GLACIERS = {
    src_region: {
        "20pct": splits[src_region]["pool_glaciers"],
    }
    for src_region in SOURCE_REGIONS
}
df_row_check = verify_row_percentage(
    pd.concat([
        res_xreg_by_source[src_region]['df_train']
        for src_region in SOURCE_REGIONS
    ]),
    FT_GLACIERS,
)
df_row_check

### Plot FT/Hold-out glaciers:

##### Switzerland:

In [ ]:
ceu_pool = FT_GLACIERS["CH"]["20pct"]
df_ceu_all = res_xreg_by_source['CH']['df_train']

ft_glaciers_by_split = {
    "20pct": ceu_pool,
}
holdout_glaciers_by_split = {
    "20pct":
    df_ceu_all.loc[~df_ceu_all["GLACIER"].isin(ceu_pool),
                   "GLACIER"].unique().tolist(),
}

data_CEU, glacier_outline_rgi, glacier_info_by_split = build_region_glacier_info_for_splits(
    cfg,
    rgi_region_id="11",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_11_CentralEurope/11_rgi60_CentralEurope.shp",
    ft_glaciers_by_split=ft_glaciers_by_split,
    split_names=("20pct", ),
    ft_label_col="FT/Hold-out glacier",
    holdout_glaciers_by_split=holdout_glaciers_by_split,
)

glacier_df_CEU_20pct = glacier_info_by_split["20pct"]

train_color = get_cmap_hex(cm.batlow, 10)[0]
palette = {"Hold-out": train_color, "FT": "#b2182b", "Excluded": "#878787"}

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_measurements_map(
    glacier_info=glacier_df_CEU_20pct,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier measurement locations Central European Alps (20pct)",
    extent=(5.8, 15, 44, 48),
    sizes=(100, 1500),
    size_legend_values=(30, 100, 1000),
    palette=palette,
    cmap_for_train=cm.batlow,
    split_col="FT/Hold-out glacier",
)

##### Norway:

In [ ]:
nor_pool = FT_GLACIERS["NOR"]["20pct"]
df_nor_all = res_xreg_by_source['NOR']['df_train']

ft_glaciers_by_split = {
    "20pct": nor_pool,
}
holdout_glaciers_by_split = {
    "20pct":
    df_nor_all.loc[~df_nor_all["GLACIER"].isin(nor_pool),
                   "GLACIER"].unique().tolist(),
}

data_NOR, glacier_outline_rgi, glacier_info_by_split = build_region_glacier_info_for_splits(
    cfg,
    rgi_region_id="08",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_08_Scandinavia/08_rgi60_Scandinavia.shp",
    ft_glaciers_by_split=ft_glaciers_by_split,
    split_names=("20pct", ),
    ft_label_col="FT/Hold-out glacier",
    holdout_glaciers_by_split=holdout_glaciers_by_split,
)

glacier_df_NOR_20pct = glacier_info_by_split["20pct"]

train_color = get_cmap_hex(cm.batlow, 10)[0]
palette = {"Hold-out": train_color, "FT": "#b2182b"}

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_measurements_map(
    glacier_info=glacier_df_NOR_20pct,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier PMB location Norway (20pct)",
    extent=(4, 24, 57, 71),
    sizes=(100, 1500),
    size_legend_values=(30, 100, 1000),
    palette=palette,
    cmap_for_train=cm.batlow,
    split_col="FT/Hold-out glacier",
    legend_ncol=2,
)

##### Alaska:

In [ ]:
ala_pool = FT_GLACIERS["ALA"]["20pct"]
df_ala_all = res_xreg_by_source['ALA']['df_train']

ft_glaciers_by_split = {
    "20pct": ala_pool,
}
holdout_glaciers_by_split = {
    "20pct":
    df_ala_all.loc[~df_ala_all["GLACIER"].isin(ala_pool),
                   "GLACIER"].unique().tolist(),
}

data_ala, glacier_outline_rgi, glacier_info_by_split = build_region_glacier_info_for_splits(
    cfg,
    rgi_region_id="01",
    outline_shp_path=cfg.dataPath + "RGI_v6/RGI_01_Alaska/01_rgi60_Alaska.shp",
    ft_glaciers_by_split=ft_glaciers_by_split,
    split_names=("20pct", ),
    ft_label_col="FT/Hold-out glacier",
    holdout_glaciers_by_split=holdout_glaciers_by_split,
)

glacier_df_ala_20pct = glacier_info_by_split["20pct"]

train_color = get_cmap_hex(cm.batlow, 10)[0]
palette = {"Hold-out": train_color, "FT": "#b2182b"}

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_measurements_map(
    glacier_info=glacier_df_ala_20pct,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier PMB location Alaska (20pct)",
    extent=(-170, -130, 55, 70),
    sizes=(100, 1000),
    size_legend_values=(30, 100, 400),
    palette=palette,
    cmap_for_train=cm.batlow,
    split_col="FT/Hold-out glacier",
)

##### Iceland:

In [ ]:
isl_pool = FT_GLACIERS["ISL"]["20pct"]
df_isl_all = res_xreg_by_source['ISL']['df_train']

ft_glaciers_by_split = {
    "20pct": isl_pool,
}
holdout_glaciers_by_split = {
    "20pct":
    df_isl_all.loc[~df_isl_all["GLACIER"].isin(isl_pool),
                   "GLACIER"].unique().tolist(),
}

data_ISL, glacier_outline_rgi, glacier_info_by_split = build_region_glacier_info_for_splits(
    cfg,
    rgi_region_id="06",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_06_Iceland/06_rgi60_Iceland.shp",
    ft_glaciers_by_split=ft_glaciers_by_split,
    split_names=("20pct", ),
    ft_label_col="FT/Hold-out glacier",
    holdout_glaciers_by_split=holdout_glaciers_by_split,
)

glacier_df_ISL_20pct = glacier_info_by_split["20pct"]

train_color = get_cmap_hex(cm.batlow, 10)[0]
palette = {"Hold-out": train_color, "FT": "#b2182b"}

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_measurements_map(
    glacier_info=glacier_df_ISL_20pct,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier PMB location Iceland (20pct)",
    extent=(-25, -11, 62, 68),
    sizes=(100, 1500),
    size_legend_values=(30, 100, 1000),
    palette=palette,
    cmap_for_train=cm.batlow,
    split_col="FT/Hold-out glacier",
)

## Transformer model
### Datasets:

In [ ]:
tl_assets_by_source = {}

REGION_GROUPS = {}

for src_region in SOURCE_REGIONS:
    print(f"\n{'='*50}")
    print(f"Building TL assets for source: {src_region}")

    # Skip target regions that overlap with the source
    src_members = set(REGION_GROUPS.get(src_region, [src_region]))
    ft_glaciers_for_src = {
        reg: splits
        for reg, splits in FT_GLACIERS.items()
        if reg not in src_members and not src_members.issubset(
            REGION_GROUPS.get(reg, [reg])) and reg != src_region
    }

    tl_assets_by_source[src_region] = build_transfer_learning_assets(
        cfg=cfg,
        res_xreg=res_xreg_by_source[src_region],
        FT_GLACIERS=ft_glaciers_for_src,
        MONTHLY_COLS=MONTHLY_COLS,
        STATIC_COLS=STATIC_COLS,
        cache_dir=BASE_DIR / "LSTM_cache_TL" / src_region,
        force_recompute=False,
        src_code=src_region,
        region_groups=REGION_GROUPS,
    )
    print(f"{src_region} -> TL asset keys:",
          list(tl_assets_by_source[src_region].keys()))

### Model parameters:

In [ ]:
gs_logs_dir = BASE_DIR / "logs/Transformer_GS"
logs_gs_transformer = pd.read_csv(
    os.path.join(gs_logs_dir, "transformer_gs_pooled_2026-05-20.csv"))

# best by validation loss
best_row = logs_gs_transformer.sort_values("valid_loss").iloc[0]
print("Best config:")
print(best_row[[
    "d_model", "nhead", "num_layers", "dim_feedforward", "dropout",
    "static_layers", "static_hidden", "static_dropout", "head_dropout", "lr",
    "weight_decay", "valid_loss", "val_rmse_a", "val_rmse_w"
]])

best_params_gs = {
    "Fm":
    int(best_row["Fm"]),
    "Fs":
    int(best_row["Fs"]),
    "d_model":
    int(best_row["d_model"]),
    "nhead":
    int(best_row["nhead"]),
    "num_layers":
    int(best_row["num_layers"]),
    "dim_feedforward":
    int(best_row["dim_feedforward"]),
    "dropout":
    float(best_row["dropout"]),
    "static_layers":
    int(best_row["static_layers"]),
    "static_hidden": (None if pd.isna(best_row["static_hidden"]) else int(
        best_row["static_hidden"])),
    "static_dropout": (None if pd.isna(best_row["static_dropout"]) else float(
        best_row["static_dropout"])),
    "head_dropout":
    float(best_row["head_dropout"]),
    "lr":
    float(best_row["lr"]),
    "weight_decay":
    float(best_row["weight_decay"]),
    "two_heads":
    False,
    "loss_spec":
    None,
    "T_max":
    32,
}

print("\nbest_params_gs:")
for k, v in best_params_gs.items():
    print(f"  {k}: {v}")


### Train or load baselines:

In [ ]:
baseline_models = {}
baseline_paths = {}

TRAIN_FLAG = True
MODEL_DATE = "2026-05-28"

for src_region in SOURCE_REGIONS:
    any_key = next(iter(tl_assets_by_source[src_region]))
    assets_src = tl_assets_by_source[src_region][any_key]

    # wrap into the shape train_or_load_one_source_model expects
    lstm_assets = {
        "ds_train": assets_src["ds_pretrain"],
        "train_idx": assets_src["pretrain_train_idx"],
        "val_idx": assets_src["pretrain_val_idx"],
    }

    model, path, info = train_or_load_one_source_model(
        cfg=cfg,
        key="defaultparams",
        lstm_assets=lstm_assets,
        best_params=best_params_gs,
        device=device,
        models_dir=BASE_DIR / "models" / src_region,
        prefix=f"transformer_{src_region}",
        train_flag=TRAIN_FLAG,
        force_retrain=False,
        epochs=150,
        date=MODEL_DATE,
        model_type="transformer",
        verbose=False,
    )

    baseline_models[src_region] = model
    baseline_paths[src_region] = path
    print(f"{src_region} baseline saved at: {path}")

### Fine tune Transformer:

In [ ]:
ft_models_by_source = {}

for src_region in SOURCE_REGIONS:
    print(f"\n{'='*50}\nFine-tuning Transformer from source: {src_region}")
    ft_models_by_source[src_region] = {}

    for exp_key, assets in tl_assets_by_source[src_region].items():
        ds_pooled_scaler = assets["ds_pretrain_scalers"]

        for strategy in ["head_only", "adapter", "full"]:
            model_ft = finetune_transformer_on_target(
                cfg=cfg,
                model_pooled=baseline_models[src_region],
                ds_ft=assets["ds_finetune"],
                ds_pooled_scaler=ds_pooled_scaler,
                device=device,
                best_params=best_params_gs,
                model_filename=str(
                    BASE_DIR / "models" / src_region /
                    f"transformer_ft_{strategy}_{exp_key}_{MODEL_DATE}.pt"),
                strategy=strategy,
                epochs=50,
                lr_factor=0.1,
                force_retrain=False,
            )
            ft_models_by_source[src_region][
                f"{exp_key}__{strategy}"] = model_ft

    print(f"{src_region} -> fine-tuned keys:",
          list(ft_models_by_source[src_region].keys()))

#### DAN:

In [ ]:
dan_models_by_source = {}

for src_region in SOURCE_REGIONS:
    print(f"\n{'='*50}\nDAN fine-tuning Transformer from source: {src_region}")
    dan_models_by_source[src_region] = {}

    for exp_key, assets in tl_assets_by_source[src_region].items():
        target_region = assets["target_code"]

        source_codes_pretrain = assets["pretrain_source_codes"]
        source_codes_ft = assets["ft_source_codes"]

        model_dan, domain_vocab = finetune_dan_transformer_on_target(
            cfg=cfg,
            model_foundation=baseline_models[src_region],
            assets_full={
                "ds_train": assets["ds_pretrain"],
                "train_idx": assets["pretrain_train_idx"],
                "val_idx": assets["pretrain_val_idx"],
            },
            ft_assets_region={
                "ds_ft": assets["ds_finetune"],
                "ft_train_idx": assets["finetune_train_idx"],
                "ft_val_idx": assets["finetune_val_idx"],
            },
            ds_pooled_scaler=assets["ds_pretrain_scalers"],
            source_codes_pretrain=source_codes_pretrain,
            source_codes_ft=source_codes_ft,
            device=device,
            best_params=best_params_gs,
            model_filename=str(BASE_DIR / "models" / src_region /
                               f"transformer_dan_{exp_key}_{MODEL_DATE}.pt"),
            dan_alpha=0.1,
            grl_lambda=1.0,
            mix_ratio_ft=1.0,
            epochs=60,
            lr_backbone=5e-5,
            lr_domain=1e-4,
            force_retrain=False,
            verbose=False,
        )
        dan_models_by_source[src_region][f"{exp_key}__dan"] = model_dan

    print(f"{src_region} -> DAN keys:",
          list(dan_models_by_source[src_region].keys()))

### Evaluate on test:

In [ ]:
# baseline_models and baseline_paths are already loaded above
# via train_or_load_CH_baseline loop — no need to reload

# Sanity check
for src_region in SOURCE_REGIONS:
    print(f"{src_region} -> baseline loaded: {baseline_paths[src_region]}")

##### 20 percent split (sparse):

In [ ]:
ax = plt.subplot(1, 1, 1)
src_region = 'CH'
target_region = 'ISL'

prefix = f"TL_{src_region}_to_"
target_regions = sorted(
    set(k[len(prefix):].rsplit("_", 1)[0]
        for k in tl_assets_by_source[src_region].keys()))
print(f"Target regions: {target_regions}")

models_xreg_for_src = {
    reg: baseline_models[src_region]
    for reg in target_regions
}
metrics, df_preds, _fig_ind, _ = evaluate_one_model_TL(
    cfg=cfg,
    model=models_xreg_for_src[target_region],
    device=device,
    tl_assets_for_key=tl_assets_by_source[src_region]
    [f'TL_{src_region}_to_{target_region}_20pct'],
    ax=ax,
    ax_xlim=None,
    ax_ylim=None,
    title=None,
    legend_fontsize=10,
    batch_size=128,
    domain_vocab=None,
    show_legend=False)

In [ ]:
ax = plt.subplot(1, 1, 1)
src_region = 'CH'
target_region = 'ISL'

prefix = f"TL_{src_region}_to_"
target_regions = sorted(
    set(k[len(prefix):].rsplit("_", 1)[0]
        for k in tl_assets_by_source[src_region].keys()))
print(f"Target regions: {target_regions}")

models_xreg_for_src = {
    reg: baseline_models[src_region]
    for reg in target_regions
}
models_total = ft_models_by_source[src_region]

metrics, df_preds, _fig_ind, _ = evaluate_one_model_TL(
    cfg=cfg,
    model=models_total[f'TL_{src_region}_to_{target_region}_20pct__adapter'],
    device=device,
    tl_assets_for_key=tl_assets_by_source[src_region]
    [f'TL_{src_region}_to_{target_region}_20pct'],
    ax=ax,
    ax_xlim=None,
    ax_ylim=None,
    title=None,
    legend_fontsize=10,
    batch_size=128,
    domain_vocab=None,
    show_legend=False)

In [ ]:
src_region = 'CH'
split_name = '20pct'

models_total = {
    **ft_models_by_source[src_region],
    **dan_models_by_source[src_region],
}

prefix = f"TL_{src_region}_to_"
target_regions = ['ISL']
print(f"Target regions: {target_regions}")

models_xreg_for_src = {
    reg: baseline_models[src_region]
    for reg in target_regions
}

df_tl_grid, preds_tl_grid, fig_tl_grid = evaluate_transfer_learning_grid(
    cfg=cfg,
    regions=target_regions,
    models_xreg_by_region=models_xreg_for_src,
    models_tl_by_key=models_total,
    tl_assets_by_key=tl_assets_by_source[src_region],
    device=device,
    split_name=split_name,
    src_code=src_region,
    save_dir=f"figures/eval_TL/{src_region}",
    strategies=["no_ft", "adapter", "dan"],
    strategy_labels={
        "no_ft": "No FT",
        "adapter": "Adapter FT",
        "dan": "Domain-Adversarial",
    },
)

plt.show()

In [ ]:
src_region = 'ISL'
split_name = '20pct'

models_total = {
    **ft_models_by_source[src_region],
    **dan_models_by_source[src_region],
}

prefix = f"TL_{src_region}_to_"
target_regions = ['CH']
print(f"Target regions: {target_regions}")

models_xreg_for_src = {
    reg: baseline_models[src_region]
    for reg in target_regions
}

df_tl_grid, preds_tl_grid, fig_tl_grid = evaluate_transfer_learning_grid(
    cfg=cfg,
    regions=target_regions,
    models_xreg_by_region=models_xreg_for_src,
    models_tl_by_key=models_total,
    tl_assets_by_key=tl_assets_by_source[src_region],
    device=device,
    split_name=split_name,
    src_code=src_region,
    save_dir=f"figures/eval_TL/{src_region}",
    strategies=["no_ft", "adapter", "dan"],
    strategy_labels={
        "no_ft": "No FT",
        "adapter": "Adapter FT",
        "dan": "Domain-Adversarial",
    },
)

plt.show()

In [ ]:
df_metrics_tl_by_source = {}

for src_region in SOURCE_REGIONS:
    print(f"\n{'='*50}")
    print(f"Evaluating source: {src_region}")

    models_total = {
        **ft_models_by_source[src_region],
        **dan_models_by_source[src_region],
    }

    prefix = f"TL_{src_region}_to_"
    target_regions = sorted(
        set(k[len(prefix):].rsplit("_", 1)[0]
            for k in tl_assets_by_source[src_region].keys()))
    print(f"Target regions: {target_regions}")

    models_xreg_for_src = {
        reg: baseline_models[src_region]
        for reg in target_regions
    }

    df_metrics_tl_by_source[src_region] = {}

    for split_name in ["20pct"]:
        df_tl_grid, preds_tl_grid, fig_tl_grid = evaluate_transfer_learning_grid(
            cfg=cfg,
            regions=target_regions,
            models_xreg_by_region=models_xreg_for_src,
            models_tl_by_key=models_total,
            tl_assets_by_key=tl_assets_by_source[src_region],
            device=device,
            split_name=split_name,
            src_code=src_region,
            save_dir=f"figures/eval_TL/{src_region}",
            strategies=["no_ft", "head_only", "full", "adapter", "dan"],
        )

        df_metrics_tl_by_source[src_region][split_name] = df_tl_grid
        plt.show()